# Train the kitchen head detector
Choose a GPU runtime. Download this repository as a source ZIP from GitHub, and agts/hairnet version 1 in YOLOv8 format from Roboflow. Upload both ZIPs when asked. The dataset is CC BY 4.0; retain attribution. No account tokens are needed in this notebook. Training does not establish kitchen safety; inspect held-out metrics and keep independent kitchen data for future evaluation.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','ultralytics==8.4.215','Pillow<13','PyYAML>=6,<7'])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'
print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()
base=Path('/content/kitchen-inputs');base.mkdir(exist_ok=True)
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as archive:
            assert all((base/n.filename).resolve().is_relative_to(base.resolve()) for n in archive.infolist())
            archive.extractall(base)
projects=[p.parent for p in base.rglob('train.py') if (p.parent/'prepare_dataset.py').exists()]
datasets=[p.parent for p in base.rglob('data.yaml') if (p.parent/'train/images').is_dir()]
assert len(projects)==1 and len(datasets)==1, 'Upload the repository source ZIP and original dataset export ZIP'
project=projects[0];source=datasets[0];data=Path('/content/hairnet-grouped')
subprocess.check_call([sys.executable,str(project/'prepare_dataset.py'),str(source),str(data)])


In [ ]:
subprocess.check_call([sys.executable,str(project/'train.py'),'--data',str(data/'data.yaml'),'--device','0','--epochs','50','--batch','8','--output','/content/kitchen-results'],cwd=project)


In [ ]:
import shutil,json
results=Path('/content/kitchen-results')
card=json.loads((results/'model_card.json').read_text())
print(json.dumps(card['test_metrics'],indent=2))
shutil.copy2(card['weights'],results/'hairnet.pt')
shutil.make_archive('/content/kitchen-trained-model','zip',results)
files.download('/content/kitchen-trained-model.zip')
